In [0]:
-- verificacion tabla silver completa
SELECT
  COUNT(*)
FROM bootcamp_de_valentin.silver.propiedades_silver;

DESCRIBE TABLE bootcamp_de_valentin.silver.propiedades_silver;

# **Modulo 1: Modelado Gold - Star Schema**

## Tablas Dimensionales


In [0]:
-- Ejercicio 1.1 
-- Crear tablas de dimensiones
-- dim_zona
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_zona (
  zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  partido STRING NOT NULL,
  region STRING NOT NULL,
  ciudad STRING,
  provincia STRING DEFAULT 'Buenos Aires',
  pais STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES( 'delta.feature.allowColumnDefaults' = 'supported' )
COMMENT 'Dimension de zonas - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_zona (partido, region, ciudad, provincia, pais)
SELECT DISTINCT
  partido,
  region,
  CASE
    WHEN region = 'capital federal' THEN 'CABA'
    ELSE 'GBA'
  END AS ciudad,
  'Buenos Aires' AS provincia,
  'Argentina' AS pais
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE partido IS NOT NULL
ORDER BY partido;

In [0]:
-- dim_tipo_operacion
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tipo_operacion(
  tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  tipo_operacion STRING NOT NULL,
  moneda STRING NOT NULL,
  categoria STRING,
  descripcion STRING,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults'= 'supported')
COMMENT 'Dimension Tipo Operacion + moneda - Star Schema' ;

INSERT INTO bootcamp_de_valentin.gold.dim_tipo_operacion (tipo_operacion, moneda, categoria, descripcion)
SELECT DISTINCT
  tipo_operacion,
  moneda,
  CASE
    WHEN tipo_operacion = 'alquiler' THEN 'residencial'
    WHEN tipo_operacion = 'venta' THEN 'residencial'
    WHEN tipo_operacion = 'alquiler_temporario' THEN 'temporal'
    ELSE 'otro'
  END AS categoria,
  CASE
    WHEN tipo_operacion = 'alquiler' THEN 'Alquiler residencial'
    WHEN tipo_operacion = 'venta' THEN 'Venta de propiedad'
    WHEN tipo_operacion = 'alquiler_temporario' THEN 'Alquiler temporal'
    ELSE 'Otro tipo'
  END AS descripcion
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE 
  tipo_operacion IS NOT NULL
  AND moneda IS NOT NULL
ORDER BY tipo_operacion, moneda;

In [0]:
-- dim_tiempo
DROP TABLE bootcamp_de_valentin.gold.dim_tiempo;

CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tiempo (
  fecha_id BIGINT,
  fecha DATE NOT NULL,
  anio INT,
  anio_mes STRING,
  anio_mes_num INT,
  mes STRING,
  mes_num INT,
  trimestre INT,
  dia_semana STRING,
  dia_semana_num INT,
  es_fin_de_semana BOOLEAN,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension tiempo - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_tiempo (fecha_id, fecha, anio, anio_mes, anio_mes_num, mes, mes_num, trimestre, dia_semana, dia_semana_num, es_fin_de_semana)
SELECT DISTINCT
  CAST( DATE_FORMAT(fecha_publicacion, 'yyyyMMdd') AS BIGINT ) AS fecha_id,
  CAST( fecha_publicacion AS DATE) AS fecha,
  YEAR(fecha_publicacion) AS anio,
  DATE_FORMAT(fecha_publicacion, 'yy-MMM') AS anio_mes,
  CAST( DATE_FORMAT(fecha_publicacion, 'yyMM') AS INT) AS anio_mes_num,
  DATE_FORMAT(fecha_publicacion, 'MMM') AS mes,
  MONTH(fecha_publicacion) AS mes_num,
  QUARTER(fecha_publicacion) AS trimestre,
  DATE_FORMAT(fecha_publicacion, 'EEE') AS dia_semana,
  EXTRACT(DAYOFWEEK_ISO FROM fecha_publicacion) AS dia_semana_num,
  CASE
    WHEN EXTRACT(DAYOFWEEK_ISO FROM fecha_publicacion) IN (6,7) THEN TRUE
    ELSE FALSE
  END AS es_fin_de_semana
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE fecha_publicacion IS NOT NULL
ORDER BY fecha;

In [0]:
-- dim_tiempo_2

-- CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tiempo (
--   fecha_id BIGINT,
--   fecha DATE NOT NULL,
--   anio INT,
--   anio_mes STRING,
--   anio_mes_num INT,
--   mes STRING,
--   mes_num INT,
--   trimestre INT,
--   dia_semana STRING,
--   dia_semana_num INT,
--   es_fin_de_semana BOOLEAN,
--   _created_at TIMESTAMP DEFAULT current_timestamp()
-- )
-- USING DELTA
-- TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
-- COMMENT 'Dimension tiempo - Star Schema';

-- INSERT INTO bootcamp_de_valentin.gold.dim_tiempo2 (fecha_id, fecha, anio, anio_mes, anio_mes_num, mes, mes_num, trimestre, dia_semana, dia_semana_num, es_fin_de_semana)


In [0]:
-- dim_caracteristicas
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_caracteristicas (
  caracteristica_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  estado STRING NOT NULL,
  cochera BOOLEAN,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Junk Dimension Caracteristicas (estado + cochera) - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
  COALESCE(estado, 'sin especificar') AS estado,
  COALESCE( cochera, false ) AS cochera
FROM bootcamp_de_valentin.silver.propiedades_silver;


In [0]:
-- dim_orientacion
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_orientacion (
  orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  orientacion STRING NOT NULL,
  tipo_orientacion STRING,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension Orientacion - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_orientacion (orientacion, tipo_orientacion)
SELECT DISTINCT
  COALESCE(orientacion, 'sin especificar') as orientacion,
  CASE 
      WHEN LOWER(orientacion) LIKE '%norte%' THEN 'norte'
      WHEN LOWER(orientacion) LIKE '%sur%' THEN 'sur'
      WHEN LOWER(orientacion) LIKE '%este%' THEN 'este'
      WHEN LOWER(orientacion) LIKE '%oeste%' THEN 'oeste'
      ELSE 'sin especificar'
  END as tipo_orientacion
FROM bootcamp_de_valentin.silver.propiedades_silver
ORDER BY orientacion;

In [0]:
-- Verificacion de dimensiones creadas
SELECT * FROM bootcamp_de_valentin.gold.dim_zona

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_tipo_operacion

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_tiempo

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_caracteristicas

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_orientacion

## Tabla de Hechos

In [0]:

DROP TABLE IF EXISTS bootcamp_de_valentin.gold.fact_propiedades;

CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.fact_propiedades (
  -- Row Hash como PK
  row_hash STRING NOT NULL,
  -- Foreing Keys
  zona_id BIGINT,
  tipo_operacion_id BIGINT,
  fecha_id BIGINT, 
  caracteristicas_id BIGINT,
  orientacion_id BIGINT,
  -- Degenerated Dimension
  url STRING,
  -- Metricas
  precio DECIMAL (15,2),
  expensas DECIMAL (15,2),
  precio_m2 DECIMAL (15,2),
  m2_totales DECIMAL (15,2),
  m2_cubiertos DECIMAL (15,2),
  ambientes INT,
  -- Fecha de actualizacion
  _refresh_timestamp TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de Hechos Alquiler Propiedades - Star Schema - PK = row_hash (MD5 de url + precio)';

INSERT INTO bootcamp_de_valentin.gold.fact_propiedades (
  row_hash, 
  zona_id, tipo_operacion_id, fecha_id, caracteristicas_id, orientacion_id,
  precio, expensas, precio_m2, m2_totales, m2_cubiertos, ambientes, 
  url
)
SELECT
  -- pk
  MD5( CONCAT_WS('|', precio, url)) AS row_hash,
  -- fks
  dz.zona_id,
  dtp.tipo_operacion_id,
  dt.fecha_id,
  dc.caracteristica_id,
  do.orientacion_id,
  -- metricas
  ps.precio,
  ps.expensas,
  ps.precio_por_m2,
  ps.m2_totales,
  ps.m2_cubiertos,
  ps.ambientes,
  -- degenerated dimension
  ps.url
FROM bootcamp_de_valentin.silver.propiedades_silver ps
LEFT JOIN bootcamp_de_valentin.gold.dim_zona dz
  ON dz.partido = ps.partido
  AND dz.region = ps.region
LEFT JOIN bootcamp_de_valentin.gold.dim_tipo_operacion dtp
  ON dtp.tipo_operacion = ps.tipo_operacion
  AND dtp.moneda = ps.moneda
LEFT JOIN bootcamp_de_valentin.gold.dim_tiempo dt
  ON dt.fecha = ps.fecha_publicacion
LEFT JOIN bootcamp_de_valentin.gold.dim_caracteristicas dc
  ON dc.estado = COALESCE( ps.estado, 'sin especificar')
  AND dc.cochera = COALESCE( ps.cochera, false)
LEFT JOIN bootcamp_de_valentin.gold.dim_orientacion do
  ON do.orientacion = COALESCE( ps.orientacion, 'sin especifica');

In [0]:
-- chequeo tabla fact
SELECT
  COUNT(*) total_Registros,
  COUNT(DISTINCT row_hash) hashes_distintos
FROM bootcamp_de_valentin.gold.fact_propiedades

# **Modulo 1: Modelado Gold - Snowflake, OBT, Galaxy**

In [0]:
-- Ejercicio 1.3
-- Snowflake: Normalizar la dimension zona en jerarquia: partido/region --> ciudad --> provincia
-- Subdimension Provincia
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_provincia_sf (
  provincia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  provincia STRING NOT NULL,
  pais STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension Provincia - Snowflake';

INSERT INTO bootcamp_de_valentin.gold.dim_provincia_sf (provincia, pais)
SELECT DISTINCT
  'Buenos Aires' AS provincia,
  'Argentina' AS pais
;

-- Subdimension Ciudad
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_ciudad_sf (
  ciudad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  provincia_id BIGINT,
  ciudad STRING NOT NULL,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension Ciudad - Snowflake';

INSERT INTO bootcamp_de_valentin.gold.dim_ciudad_sf (ciudad, provincia_id)
SELECT DISTINCT
  CASE
    WHEN region = 'capital federal' THEN 'CABA'
    ELSE 'GBA'
  END AS ciudad,
  1 as provincia_id
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE
  region IS NOT NULL
;

-- Subdimension Zona
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_zona_sf (
  zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  ciudad_id BIGINT,
  partido STRING NOT NULL,
  region STRING NOT NULL,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension Zona - Snowflake';

INSERT INTO bootcamp_de_valentin.gold.dim_zona_sf (partido, region, ciudad_id)
SELECT DISTINCT
  ps.partido,
  ps.region,
  dc.ciudad_id
FROM bootcamp_de_valentin.silver.propiedades_silver ps
JOIN bootcamp_de_valentin.gold.dim_ciudad_sf dc ON
  dc.ciudad = CASE
                WHEN ps.region = 'capital federal' THEN 'CABA'
                ELSE 'GBA'
              END
WHERE
  ps.partido IS NOT NULL
;

In [0]:
-- Ejercicio 1.4
-- OBT
CREATE TABLE bootcamp_de_valentin.gold.obt_propiedades_completa(
  propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  precio DECIMAL(15,2),
  moneda STRING,
  expensas DECIMAL(15,2),
  ambientes INT,
  m2_totales DECIMAL(15,2),
  m2_cubiertos DECIMAL(15,2),
  antiguedad INT,
  cochera BOOLEAN,
  precio_por_m2 DECIMAL(15,2),
  partido STRING,
  region STRING,
  ciudad STRING,
  provincia STRING DEFAULT 'Buenos Aires',
  pais STRING DEFAULT 'Argentina',
  tipo_operacion STRING,
  categoria_operacion STRING,
  estado STRING,
  categoria_estado STRING,
  orientacion STRING,
  tipo_orientacion STRING,
  fecha_publicacion DATE,
  anio INT,
  mes INT,
  trimestre INT,
  dia_semana STRING,
  es_fin_de_semana BOOLEAN,
  segmento_precio STRING,
  rango_metros STRING,
  rango_ambientes STRING,
  tiene_expensas BOOLEAN,
  ratio_m2_cubiertos_totales DECIMAL(5,2),
  url STRING,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'One Big Table - Propiedades completas denormalizadas';

INSERT INTO bootcamp_de_valentin.gold.obt_propiedades_completa (
  precio, moneda, expensas, ambientes,
  m2_totales, m2_cubiertos,
  antiguedad, cochera, precio_por_m2,
  partido, region, ciudad, provincia, pais,
  tipo_operacion, categoria_operacion,
  estado, categoria_estado,
  orientacion, tipo_orientacion,
  fecha_publicacion, anio, mes, trimestre, dia_semana, es_fin_de_semana,
  segmento_precio, rango_metros, rango_ambientes,
  tiene_expensas, ratio_m2_cubiertos_totales,
  url
)
SELECT 
    sp.precio, sp.moneda, sp.expensas, sp.ambientes,
    sp.m2_totales, sp.m2_cubiertos,
    sp.antiguedad, sp.cochera, sp.precio_por_m2,
    sp.partido,
    sp.region,
    CASE WHEN sp.region = 'capital federal' THEN 'CABA' ELSE 'GBA' END as ciudad,
    'Buenos Aires' as provincia, 'Argentina' as pais,
    sp.tipo_operacion,
    CASE 
        WHEN sp.tipo_operacion = 'alquiler' THEN 'residencial'
        WHEN sp.tipo_operacion = 'venta' THEN 'residencial'
        WHEN sp.tipo_operacion = 'alquiler_temporario' THEN 'temporal'
        ELSE 'otro'
    END as categoria_operacion,
    sp.estado,
    CASE 
        WHEN LOWER(sp.estado) LIKE '%nuevo%' THEN 'nuevo'
        WHEN LOWER(sp.estado) LIKE '%usado%' THEN 'usado'
        WHEN LOWER(sp.estado) LIKE '%remodelado%' THEN 'remodelado'
        ELSE 'sin especificar'
    END as categoria_estado,
    COALESCE(sp.orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(sp.orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(sp.orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(sp.orientacion) LIKE '%este%' THEN 'este'
        WHEN LOWER(sp.orientacion) LIKE '%oeste%' THEN 'oeste'
        ELSE 'sin especificar'
    END as tipo_orientacion,
    sp.fecha_publicacion,
    YEAR(sp.fecha_publicacion) as anio, MONTH(sp.fecha_publicacion) as mes,
    QUARTER(sp.fecha_publicacion) as trimestre,
    CASE DAYOFWEEK(sp.fecha_publicacion)
        WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Lunes' WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles' WHEN 5 THEN 'Jueves' WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END as dia_semana,
    DAYOFWEEK(sp.fecha_publicacion) IN (1, 7) as es_fin_de_semana,
    CASE 
        WHEN sp.moneda = 'ARS' AND sp.precio < 300000 THEN 'bajo'
        WHEN sp.moneda = 'ARS' AND sp.precio < 500000 THEN 'medio'
        WHEN sp.moneda = 'ARS' AND sp.precio < 800000 THEN 'alto'
        WHEN sp.moneda = 'ARS' THEN 'premium'
        WHEN sp.moneda = 'USD' AND sp.precio < 500 THEN 'bajo'
        WHEN sp.moneda = 'USD' AND sp.precio < 1000 THEN 'medio'
        WHEN sp.moneda = 'USD' AND sp.precio < 2000 THEN 'alto'
        WHEN sp.moneda = 'USD' THEN 'premium'
        ELSE 'sin clasificar'
    END as segmento_precio,
    CASE 
        WHEN sp.m2_totales < 50 THEN 'pequeño'
        WHEN sp.m2_totales < 100 THEN 'mediano'
        WHEN sp.m2_totales < 200 THEN 'grande'
        ELSE 'muy grande'
    END as rango_metros,
    CASE 
        WHEN sp.ambientes = 1 THEN 'mono'
        WHEN sp.ambientes BETWEEN 2 AND 3 THEN '2-3'
        WHEN sp.ambientes >= 4 THEN '4+'
        ELSE 'sin especificar'
    END as rango_ambientes,
    sp.expensas IS NOT NULL AND sp.expensas > 0 as tiene_expensas,
    ROUND(sp.m2_totales / NULLIF(sp.m2_totales, 0), 2) as ratio_m2_cubiertos_totales,
    sp.url
FROM bootcamp_de_valentin.silver.propiedades_silver sp


In [0]:
-- Ejercicio 1.5
-- Galaxy Schema
CREATE TABLE bootcamp_de_valentin.gold.fact_consultas (
  consulta_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  zona_id BIGINT,
  fecha_id BIGINT,
  propiedad_url STRING,
  tipo_consulta STRING,
  cantidad_consulta INT DEFAULT 1,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Hechos Consultas - Galaxy Schema';

INSERT INTO bootcamp_de_valentin.gold.fact_consultas (
  zona_id, fecha_id, propiedad_url, tipo_consulta, cantidad_consulta
)
SELECT 
  dz.zona_id AS zona_id,
  dt.fecha_id AS fecha_id,
  ps.url AS propiedad_url,
  CASE (RANDOM() * 4)::INT
    WHEN 0 THEN 'vista'
    WHEN 1 THEN 'contacto'
    WHEN 2 THEN 'favorito'
    ELSE 'compartir'
  END as tipo_consulta,
  (RANDOM() * 10 + 1)::INT as cantidad_consultas
FROM bootcamp_de_valentin.silver.propiedades_silver ps
JOIN bootcamp_de_valentin.gold.dim_zona dz ON dz.partido = ps.partido AND dz.region = ps.region
JOIN bootcamp_de_valentin.gold.dim_tiempo dt ON dt.fecha_id = DATE_FORMAT(ps.fecha_publicacion, 'yyyyMMdd')
WHERE
  ps.precio_por_m2 < (
    SELECT PERCENTILE_APPROX(precio_por_m2, 0.5) 
    FROM bootcamp_de_valentin.silver.propiedades_silver
);

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.fact_consultas

In [0]:
-- Análisis cruzado entre hechos
SELECT 
    dz.partido, dz.region,
    COUNT(DISTINCT fp.row_hash) as total_propiedades,
    COUNT(DISTINCT fc.consulta_id) as total_consultas,
    SUM(fc.cantidad_consulta) as cantidad_total_consultas,
    ROUND(AVG(fp.precio_m2), 2) as precio_m2_promedio,
    ROUND(SUM(fc.cantidad_consulta) * 1.0 / NULLIF(COUNT(DISTINCT fp.row_hash), 0), 2) as consultas_por_propiedad
FROM bootcamp_de_valentin.gold.dim_zona dz
LEFT JOIN bootcamp_de_valentin.gold.fact_propiedades fp ON dz.zona_id = fp.zona_id
LEFT JOIN bootcamp_de_valentin.gold.fact_consultas fc ON dz.zona_id = fc.zona_id
GROUP BY dz.partido, dz.region
HAVING COUNT(DISTINCT fp.row_hash) > 0
ORDER BY consultas_por_propiedad DESC
LIMIT 20;

In [0]:
-- Ejercicio 1.6
-- Precio promedio por partido y tipo de operacion
-- Star Schema
SELECT
  dz.partido,
  dto.tipo_operacion,
  AVG(fp.precio_m2) as precio_m2_promedio
FROm bootcamp_de_valentin.gold.fact_propiedades fp
LEFT JOIN bootcamp_de_valentin.gold.dim_zona dz ON dz.zona_id = fp.zona_id
LEFT JOIN bootcamp_de_valentin.gold.dim_tipo_operacion dto ON dto.tipo_operacion_id = fp.tipo_operacion_id
GROUP BY dz.partido, dto.tipo_operacion
ORDER BY dz.partido, dto.tipo_operacion;

-- Snowflake
SELECT
  dz.partido,
  dto.tipo_operacion,
  AVG(fp.precio_m2) as precio_m2_promedio
FROm bootcamp_de_valentin.gold.fact_propiedades fp
LEFT JOIN bootcamp_de_valentin.gold.dim_zona_sf dz ON dz.zona_id = fp.zona_id
LEFT JOIN bootcamp_de_valentin.gold.dim_tipo_operacion dto ON dto.tipo_operacion_id = fp.tipo_operacion_id
GROUP BY dz.partido, dto.tipo_operacion
ORDER BY dz.partido, dto.tipo_operacion;

-- OBT
SELECT
  partido,
  tipo_operacion,
  AVG(precio_por_m2) as precio_m2_promedio
FROM bootcamp_de_valentin.gold.obt_propiedades_completa
GROUP BY partido, tipo_operacion
ORDER BY partido, tipo_operacion;